In [1]:
import os
import json
import re
import unicodedata
from pathlib import Path
import pandas as pd
import spacy
import pdfplumber
from bs4 import BeautifulSoup

In [2]:
try:
    nlp = spacy.load("es_core_news_sm", disable=["ner", "lemmatizer", "textcat"])
except OSError:
    import spacy.cli
    spacy.cli.download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm", disable=["ner", "lemmatizer", "textcat"])

nlp.max_length = 5000000

# Limpieza de datos

In [3]:
def limpiar_texto(texto: str) -> str:
    if not texto: return ""
    texto = unicodedata.normalize("NFC", texto)
    texto = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]", "", texto)
    texto = texto.replace("\xa0", " ").replace("\r", "\n")
    
    # Reglas para pies/encabezados
    texto = re.sub(r"(?i)\bpágena\s+\d+(\s+de\s+\d+)?\b", "", texto)
    texto = re.sub(r"(?i)\bpage\s+\d+(\s+of\s+\d+)?\b", "", texto)
    texto = re.sub(r"^\s*-\s*\d+\s*-\s*$", "", texto, flags=re.MULTILINE)
    
    # --- ¡NUEVAS REGLAS CONTRA ÍNDICES (TOC) Y RUIDO! ---
    # 1. Eliminar secuencias largas de puntos (ej: ". . . . ." o "........")
    texto = re.sub(r"(?:\.\s*){3,}", " ", texto)
    
    # 2. Normalizar saltos de línea extraños (une líneas que se cortaron mal)
    # Si una línea termina en minúscula y la siguiente empieza sin punto, las une.
    texto = re.sub(r"([a-z,])\n([A-Za-z])", r"\1 \2", texto)
    
    # Normalización final de espacios
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    
    return texto.strip()

Extracción multiformato

In [4]:
# =========================================================
# PASO 3: EXTRACCIÓN (Actualizado para JSON tipo Lista)
# =========================================================
def extraer_texto_de_archivo(ruta_archivo: Path):
    ext = ruta_archivo.suffix.lower()
    
    if ext in [".csv", ".xlsx", ".xls"]:
        filas_convertidas = []
        if ext == ".csv":
            try:
                sep = None
                engine = "python"
                on_bad_lines = "skip"
                df = pd.read_csv(ruta_archivo, sep = None, engine = "python", encoding="utf-8-sig", on_bad_lines = "skip")
            except Exception:
                df = pd.read_csv(ruta_archivo, sep = None, engine = "python", encoding = "latin-1", on_bad_lines = "skip")

            hojas = [("Sheet1", df)]
        else:
            excel = pd.ExcelFile(ruta_archivo)
            hojas = [(hoja, pd.read_excel(excel, sheet_name=hoja)) for hoja in excel.sheet_names]
        for nombre_hoja, df_hoja in hojas:
            columnas = [str(c).strip() for c in df_hoja.columns]
            for _, fila in df_hoja.iterrows():
                datos_fila = [f"{col}: {str(fila[col]).strip()}" for col in columnas if pd.notna(fila[col]) and str(fila[col]).strip()]
                if datos_fila:
                    prefijo = f"[Pestaña: {nombre_hoja}] " if ext != ".csv" else ""
                    filas_convertidas.append(prefijo + " | ".join(datos_fila))
        return filas_convertidas
    
    elif ext == ".json":
        with open(ruta_archivo, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        # Función auxiliar para extraer texto de un diccionario individual
        def extraer_de_diccionario(d):
            if not isinstance(d, dict):
                return str(d)
            titulo = str(d.get("title", "")).strip()
            parrafos = []
            if "body_text" in d and isinstance(d["body_text"], str):
                parrafos.append(d["body_text"])
            elif "body_paragraphs" in d and isinstance(d["body_paragraphs"], list):
                for p in d["body_paragraphs"]:
                    if isinstance(p, str) and p.strip(): parrafos.append(p.strip())
                    elif isinstance(p, dict) and "text" in p: parrafos.append(str(p["text"]).strip())
            return (f"Título: {titulo}\n\n" if titulo else "") + "\n\n".join(parrafos)

        # Evaluar si la raíz es un diccionario o una lista
        if isinstance(data, dict):
            return extraer_de_diccionario(data)
        elif isinstance(data, list):
            # Si es una lista, extraemos el texto de cada elemento y lo unimos
            textos_lista = [extraer_de_diccionario(item) for item in data]
            return "\n\n".join(textos_lista)
            
    elif ext == ".pdf":
        paginas_texto = []
        try:
            with pdfplumber.open(ruta_archivo) as pdf:
                for pag in pdf.pages:
                    t = pag.extract_text()
                    if t: paginas_texto.append(t)
        except Exception: pass
        return "\n".join(paginas_texto)
        
    elif ext in [".html", ".htm"]:
        try:
            with open(ruta_archivo, "r", encoding="utf-8") as f:
                soup = BeautifulSoup(f, "html.parser")
            for script in soup(["script", "style"]): script.extract()
            return soup.get_text(separator="\n")
        except Exception: return ""
        
    elif ext in [".txt", ".md"]:
        with open(ruta_archivo, "r", encoding="utf-8") as f:
            return f.read()

    elif ext in [".png", ".jpg", ".jpeg", ".tiff", ".bmp"]:
        texto_imagen = f"Imagen: {ruta_archivo.name}. "
        try:
            from PIL import Image
            import pytesseract
            
            # Si en Windows te sale error de ruta de Tesseract, descomenta y ajusta esta línea:
            # pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
            
            img = Image.open(ruta_archivo)
            # Intentamos detectar texto en español e inglés
            texto_extraido = pytesseract.image_to_string(img, lang='spa+eng').strip()
            
            if texto_extraido:
                texto_imagen += f"Texto detectado: {texto_extraido}"
        except Exception as e:
            print(f"⚠️ Aviso OCR en {ruta_archivo.name}: {e}")
            
        return texto_imagen

    # =========================================================
    # CASO G: MAPAS PBF (Extracción de etiquetas semánticas)
    # =========================================================
    elif ext == ".pbf":
        texto_mapa = f"Mapa PBF: {ruta_archivo.name}. "
        try:
            import osmium
            
            # Creamos un manejador ligero que solo extrae nombres de lugares
            class EtiquetaHandler(osmium.SimpleHandler):
                def __init__(self):
                    super().__init__()
                    self.lugares = []

                def procesar(self, tags):
                    # Solo guardamos cosas que tengan un nombre (calles, edificios, ciudades)
                    if 'name' in tags and len(self.lugares) < 500: # Límite para no desbordar RAM
                        self.lugares.append(tags['name'])

                def node(self, n): self.procesar(n.tags)
                def way(self, w): self.procesar(w.tags)

            handler = EtiquetaHandler()
            handler.apply_file(str(ruta_archivo))
            
            if handler.lugares:
                # Quitamos duplicados y unimos los lugares en un párrafo legible
                lugares_unicos = list(set(handler.lugares))
                texto_mapa += f"Entidades geográficas presentes: {', '.join(lugares_unicos)}."
                
        except Exception as e:
            print(f"⚠️ Aviso PBF en {ruta_archivo.name}: {e}")
            
        return texto_mapa

    return ""

Función de Chunking

In [5]:
# =========================================================
# PASO 4: CHUNKING (Actualizado con URL metadata)
# =========================================================
def crear_fragmentos(texto: str, doc_id: str, fuente: str, formato: str, fenomeno: int, idioma: str, url: str = "", max_tokens: int = 250, inicio_idx: int = 0):
    doc_spacy = nlp(texto)
    oraciones = [oracion.text.strip() for oracion in doc_spacy.sents if oracion.text.strip()]
    
    fragmentos = []
    oraciones_actuales = []
    tokens_acumulados = 0
    posicion_chunk = inicio_idx

    for oracion in oraciones:
        tokens_oracion = len(oracion.split())
        if tokens_acumulados + tokens_oracion > max_tokens and oraciones_actuales:
            texto_chunk = " ".join(oraciones_actuales)
            if len(texto_chunk.split()) >= 10:
                chunk_data = {
                    "doc_id": doc_id, "chunk_id": f"{doc_id}-chunk-{posicion_chunk:04d}",
                    "fuente": fuente, "formato": formato, "fenomeno": fenomeno,
                    "idioma": idioma, "posicion": posicion_chunk,
                    "num_tokens": tokens_acumulados, "texto": texto_chunk
                }
                if url:
                    chunk_data["url"] = url
                fragmentos.append(chunk_data)
                posicion_chunk += 1
                
            oraciones_actuales = [oraciones_actuales[-1]]
            tokens_acumulados = len(oraciones_actuales[0].split())
        else:
            oraciones_actuales.append(oracion)
            tokens_acumulados += tokens_oracion

    if oraciones_actuales:
        texto_chunk = " ".join(oraciones_actuales)
        if len(texto_chunk.split()) >= 10:
            chunk_data = {
                "doc_id": doc_id, "chunk_id": f"{doc_id}-chunk-{posicion_chunk:04d}",
                "fuente": fuente, "formato": formato, "fenomeno": fenomeno,
                "idioma": idioma, "posicion": posicion_chunk,
                "num_tokens": tokens_acumulados, "texto": texto_chunk
            }
            if url:
                chunk_data["url"] = url
            fragmentos.append(chunk_data)
            
    return fragmentos

# =========================================================
# PASO 5: PROCESADOR (Actualizado para rescatar URLs de JSON)
# =========================================================
def procesar_carpeta(ruta_carpeta: str, fenomeno: int, archivo_salida: str):
    base_path = Path(ruta_carpeta)
    if not base_path.exists():
        print(f"⚠️ La ruta {ruta_carpeta} no existe. Saltando...")
        return

    todos_los_chunks = []
    contador_documentos = 1
    print(f"\n🚀 Iniciando Fenómeno {fenomeno} en: {ruta_carpeta}")

    for ruta_archivo in base_path.rglob('*'):
        if ruta_archivo.is_file() and not ruta_archivo.name.startswith('.'):
            partes_ruta = ruta_archivo.relative_to(base_path).parts
            organizacion = partes_ruta[0] if len(partes_ruta) > 0 else "ORG"
            doc_id = f"DOC-F{fenomeno}-{organizacion[:8].upper()}-{contador_documentos:04d}"
            ext = ruta_archivo.suffix.lower().replace(".", "")
            fuente = ruta_archivo.name

            url_documento = ""
            if ext == "json":
                try:
                    with open(ruta_archivo, "r", encoding="utf-8") as f:
                        temp_data = json.load(f)
                        if isinstance(temp_data, dict):
                            url_documento = str(temp_data.get("url", "")).strip()
                        elif isinstance(temp_data, list) and len(temp_data) > 0 and isinstance(temp_data[0], dict):
                            url_documento = str(temp_data[0].get("url", "")).strip()
                except Exception:
                    pass

            contenido = extraer_texto_de_archivo(ruta_archivo)
            texto_para_deteccion = " ".join(contenido[:5]) if isinstance(contenido, list) else contenido
            idioma_doc = detectar_idioma(texto_para_deteccion)

            print(f"📄 Procesando [{doc_id}] (Idioma: {idioma_doc}): {fuente}")

            if isinstance(contenido, list):
                idx_chunk = 0
                for fila_texto in contenido:
                    fila_limpia = limpiar_texto(fila_texto)
                    if fila_limpia:
                        chunks_fila = crear_fragmentos(fila_limpia, doc_id, fuente, ext, fenomeno, idioma_doc, url=url_documento, inicio_idx=idx_chunk)
                        todos_los_chunks.extend(chunks_fila)
                        idx_chunk += len(chunks_fila)
            elif isinstance(contenido, str) and contenido.strip():
                texto_limpio = limpiar_texto(contenido)
                chunks_doc = crear_fragmentos(texto_limpio, doc_id, fuente, ext, fenomeno, idioma_doc, url=url_documento)
                todos_los_chunks.extend(chunks_doc)

            contador_documentos += 1

    with open(archivo_salida, "a", encoding="utf-8") as f:
        for chunk in todos_los_chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    print(f"✅ Fenómeno {fenomeno} completado. {len(todos_los_chunks)} fragmentos extraídos.")


In [6]:
from langdetect import detect, DetectorFactory

# Esto garantiza que el detector sea determinista y siempre dé el mismo resultado
DetectorFactory.seed = 0 

def detectar_idioma(texto: str) -> str:
    """Detecta el idioma mayoritario de un texto. Devuelve el código ISO (ej: 'es', 'en', 'fr')."""
    if not texto or len(texto.strip()) < 10:
        return "unknown"
    try:
        # Detectamos basados en los primeros 1000 caracteres para hacerlo súper rápido
        return detect(texto[:1000])
    except Exception:
        return "unknown"

In [7]:
# =========================================================
# PASO 5: PROCESADOR (Actualizado para rescatar URLs de JSON)
# =========================================================
def procesar_carpeta(ruta_carpeta: str, fenomeno: int, archivo_salida: str):
    base_path = Path(ruta_carpeta)
    if not base_path.exists():
        print(f"⚠️ La ruta {ruta_carpeta} no existe. Saltando...")
        return

    todos_los_chunks = []
    contador_documentos = 1
    print(f"\n🚀 Iniciando Fenómeno {fenomeno} en: {ruta_carpeta}")

    for ruta_archivo in base_path.rglob('*'):
        if ruta_archivo.is_file() and not ruta_archivo.name.startswith('.'):
            partes_ruta = ruta_archivo.relative_to(base_path).parts
            organizacion = partes_ruta[0] if len(partes_ruta) > 0 else "ORG"
            doc_id = f"DOC-F{fenomeno}-{organizacion[:8].upper()}-{contador_documentos:04d}"
            ext = ruta_archivo.suffix.lower().replace(".", "")
            fuente = ruta_archivo.name

            # --- NUEVO: Rescate rápido de URL solo para JSON ---
            url_documento = ""
            if ext == "json":
                try:
                    with open(ruta_archivo, "r", encoding="utf-8") as f:
                        temp_data = json.load(f)
                        if isinstance(temp_data, dict):
                            url_documento = str(temp_data.get("url", "")).strip()
                        elif isinstance(temp_data, list) and len(temp_data) > 0 and isinstance(temp_data[0], dict):
                            url_documento = str(temp_data[0].get("url", "")).strip()
                except Exception:
                    pass
            # ---------------------------------------------------

            contenido = extraer_texto_de_archivo(ruta_archivo)
            texto_para_deteccion = " ".join(contenido[:5]) if isinstance(contenido, list) else contenido
            idioma_doc = detectar_idioma(texto_para_deteccion)

            print(f"📄 Procesando [{doc_id}] (Idioma: {idioma_doc}): {fuente}")

            if isinstance(contenido, list):
                idx_chunk = 0
                for fila_texto in contenido:
                    fila_limpia = limpiar_texto(fila_texto)
                    if fila_limpia:
                        # Pasamos la URL a crear_fragmentos
                        chunks_fila = crear_fragmentos(fila_limpia, doc_id, fuente, ext, fenomeno, idioma_doc, url=url_documento, inicio_idx=idx_chunk)
                        todos_los_chunks.extend(chunks_fila)
                        idx_chunk += len(chunks_fila)
            elif isinstance(contenido, str) and contenido.strip():
                texto_limpio = limpiar_texto(contenido)
                # Pasamos la URL a crear_fragmentos
                chunks_doc = crear_fragmentos(texto_limpio, doc_id, fuente, ext, fenomeno, idioma_doc, url=url_documento)
                todos_los_chunks.extend(chunks_doc)

            contador_documentos += 1

    with open(archivo_salida, "a", encoding="utf-8") as f:
        for chunk in todos_los_chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    print(f"✅ Fenómeno {fenomeno} completado. {len(todos_los_chunks)} fragmentos extraídos.")


In [8]:
# Llama a la función orquestadora apuntando a tu carpeta
#procesar_carpeta(ruta_carpeta="./F2_Seguridad_Entorno_Espacial", fenomeno=2, archivo_salida="metadata.jsonl")

In [9]:
def procesar_todo_el_reto(lista_temas: list, archivo_maestro: str = "metadata_unificada.jsonl"):
    """
    Función iterativa que recorre los 3 temas y unifica los metadatos.
    """
    print(f"🧹 Inicializando archivo maestro: {archivo_maestro}")
    
    # 1. Creamos el archivo maestro en blanco (borra pruebas anteriores)
    with open(archivo_maestro, "w", encoding="utf-8") as f:
        pass 
    
    # 2. Iteramos sobre cada tema de la lista
    for tema in lista_temas:
        ruta = tema["ruta"]
        fenomeno_id = tema["id"]
        
        print(f"\n==================================================")
        print(f"🌟 INICIANDO EXTRACCIÓN: FENÓMENO {fenomeno_id}")
        print(f"==================================================")
        
        # Llamamos a TU función procesar_carpeta
        procesar_carpeta(ruta_carpeta=ruta, fenomeno=fenomeno_id, archivo_salida=archivo_maestro)
        
    print(f"\n🎉 ¡PROCESO TOTAL FINALIZADO! Toda la base está en '{archivo_maestro}'.")

# =========================================================
# EJECUCIÓN DEL COMPLEMENTO
# =========================================================
# Solo debes ajustar las rutas a como se llamen tus carpetas
mis_3_carpetas = [
    {"ruta": "./F1_IA_y_Capacidades_Estrategicas", "id": 1},
    {"ruta": "./F2_Seguridad_Entorno_Espacial", "id": 2},
    {"ruta": "./F3_Dinamicas_Territoriales", "id": 3}
]

procesar_todo_el_reto(lista_temas=mis_3_carpetas, archivo_maestro="metadata.jsonl")


🧹 Inicializando archivo maestro: metadata.jsonl

🌟 INICIANDO EXTRACCIÓN: FENÓMENO 1

🚀 Iniciando Fenómeno 1 en: ./F1_IA_y_Capacidades_Estrategicas
📄 Procesando [DOC-F1-CENIA-0001] (Idioma: pt): CENIA_dpj-500674644740-325343.pdf
📄 Procesando [DOC-F1-CENIA-0002] (Idioma: es): CENIA_resolucion-de-subsidio-innova-exenta-electronica-24cvi-26468.pdf
📄 Procesando [DOC-F1-CENIA-0003] (Idioma: en): CENIA_paper-112.pdf
📄 Procesando [DOC-F1-CENIA-0004] (Idioma: es): CENIA_resumen-anid.pdf
📄 Procesando [DOC-F1-CENIA-0005] (Idioma: es): CENIA_memoria-cenia-2022.pdf
📄 Procesando [DOC-F1-CENIA-0006] (Idioma: en): CENIA_estados-resultados-cenia-historico-1-1.pdf
📄 Procesando [DOC-F1-CENIA-0007] (Idioma: en): CENIA_balance-2024-cenia-firmado.pdf
📄 Procesando [DOC-F1-CENIA-0008] (Idioma: en): CENIA_balance-2022-firmado-1.pdf
📄 Procesando [DOC-F1-CENIA-0009] (Idioma: es): CENIA_memoria-anual-cenia-2023.pdf
📄 Procesando [DOC-F1-CENIA-0010] (Idioma: es): CENIA_convenio-eqy250001-cenia-eqy250001-30264-compr

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


📄 Procesando [DOC-F1-CSET_GEO-0085] (Idioma: en): CSET_center-for-security-and-emerging-technology-23.pdf
📄 Procesando [DOC-F1-CSET_GEO-0086] (Idioma: en): CSET_center-for-security-and-emerging-technology-37.pdf
📄 Procesando [DOC-F1-CSET_GEO-0087] (Idioma: en): CSET_center-for-security-and-emerging-technology-21.pdf
📄 Procesando [DOC-F1-CSET_GEO-0088] (Idioma: en): CSET_center-for-security-and-emerging-technology-35.pdf
📄 Procesando [DOC-F1-CSET_GEO-0089] (Idioma: en): CSET_center-for-security-and-emerging-technology-34.pdf
📄 Procesando [DOC-F1-CSET_GEO-0090] (Idioma: nl): CSET_center-for-security-and-emerging-technology-20.pdf
📄 Procesando [DOC-F1-CSET_GEO-0091] (Idioma: en): CSET_center-for-security-and-emerging-technology-46.pdf
📄 Procesando [DOC-F1-CSET_GEO-0092] (Idioma: en): CSET_center-for-security-and-emerging-technology-44.pdf
📄 Procesando [DOC-F1-CSET_GEO-0093] (Idioma: en): CSET_center-for-security-and-emerging-technology-45.pdf
📄 Procesando [DOC-F1-CSET_GEO-0094] (Idioma: e

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

📄 Procesando [DOC-F3-SIPRI-0134] (Idioma: en): SIPRI_stockholm-forum-session-a-decade-of-youth-peace-and-security-fostering-intergenerational-a.pdf
📄 Procesando [DOC-F3-SIPRI-0135] (Idioma: en): SIPRI_2604-milex-2025.pdf
📄 Procesando [DOC-F3-SIPRI-0136] (Idioma: en): SIPRI_fs-2512-top-100-2024.pdf
📄 Procesando [DOC-F3-SIPRI-0137] (Idioma: en): SIPRI_0626-transitions-just-future.pdf
📄 Procesando [DOC-F3-SIPRI-0138] (Idioma: en): SIPRI_centerfold-east-a3.pdf
📄 Procesando [DOC-F3-SIPRI-0139] (Idioma: en): SIPRI_gendered-dimensions-of-climate-related-security-risks.pdf
📄 Procesando [DOC-F3-SIPRI-0140] (Idioma: en): SIPRI_0326-preventing-biological-weapons-proliferation-0.pdf
📄 Procesando [DOC-F3-SIPRI-0141] (Idioma: en): SIPRI_end-of-mission-ville-savoranta.pdf
📄 Procesando [DOC-F3-SIPRI-0142] (Idioma: zh-cn): SIPRI_0226-rpp-chinese-perspectives-chinese.pdf
📄 Procesando [DOC-F3-SIPRI-0143] (Idioma: en): SIPRI_pb-2601-china-and-the-changing-international-dev-landscape.pdf
📄 Procesando [DOC-

In [10]:
# 1. Creamos un texto difícil con abreviaturas (EE.UU., Dra., org.) y puntos decimales.
texto_prueba = (
    "El director de la org. dijo que la inversión inicial fue de 15.5 millones de dólares. "
    "Esto representa un hito importante para los EE.UU. en el nuevo sector aeroespacial. "
    "Además, la Dra. Gómez confirmó que el primer lanzamiento oficial será en el año 2027. "
    "Necesitamos asegurar que esta última oración no quede cortada a la mitad por culpa del límite."
)

print("🧹 Limpiando texto...")
texto_limpio = limpiar_texto(texto_prueba)

print("✂️ Generando fragmentos (Límite forzado a 30 tokens)...\n")
# Forzamos max_tokens=30 para ver cómo decide cortar el texto
chunks_prueba = crear_fragmentos(
    texto=texto_limpio, 
    doc_id="DOC-TEST-001", 
    fuente="prueba_local", 
    formato="txt", 
    fenomeno=2, 
    idioma="es", 
    max_tokens=30
)

# 2. Imprimimos el resultado de forma legible para inspección visual
for i, chunk in enumerate(chunks_prueba):
    print(f"--- Chunk {i+1} (Tokens: {chunk['num_tokens']}) ---")
    print(f"Texto: {chunk['texto']}\n")

🧹 Limpiando texto...
✂️ Generando fragmentos (Límite forzado a 30 tokens)...

--- Chunk 1 (Tokens: 29) ---
Texto: El director de la org. dijo que la inversión inicial fue de 15.5 millones de dólares. Esto representa un hito importante para los EE.UU. en el nuevo sector aeroespacial.

--- Chunk 2 (Tokens: 29) ---
Texto: Esto representa un hito importante para los EE.UU. en el nuevo sector aeroespacial. Necesitamos asegurar que esta última oración no quede cortada a la mitad por culpa del límite.



In [11]:
import json

archivo = "metadata.jsonl"
lineas_a_leer = 3  # Cambia este número para ver más o menos fragmentos

print(f"👀 Revisando las primeras {lineas_a_leer} líneas de {archivo}:\n")

try:
    with open(archivo, "r", encoding="utf-8") as f:
        for i in range(lineas_a_leer):
            linea = f.readline()
            if not linea:
                break
            # Convertimos la línea a un diccionario de Python para imprimirlo bonito
            datos = json.loads(linea)
            print(f"--- Fragmento {i+1} ---")
            print(json.dumps(datos, indent=4, ensure_ascii=False))
            print("\n")
except Exception as e:
    print(f"Error al leer el archivo: {e}")

👀 Revisando las primeras 3 líneas de metadata.jsonl:

--- Fragmento 1 ---
{
    "doc_id": "DOC-F1-CENIA-0001",
    "chunk_id": "DOC-F1-CENIA-0001-chunk-0000",
    "fuente": "CENIA_dpj-500674644740-325343.pdf",
    "formato": "pdf",
    "fenomeno": 1,
    "idioma": "pt",
    "posicion": 0,
    "num_tokens": 222,
    "texto": "SERVICIO DE REGISTRO FOLIO : 500674644740\nCIVIL E IDENTIFICACIÓN\nCódigo Verificación:\nea7902cd76b6\nREPUBLICA DE CHILE 500674644740\nCERTIFICADO DE DIRECTORIO DE\nPERSONA JURÍDICA SIN FINES DE LUCRO\nFecha Emisión 16-01-2026\nDATOS PERSONA JURÍDICA\nINSCRIPCIÓN : N°325343 con fecha 08-02-2022. NOMBRE PJ : CORPORACION CENTRO NACIONAL DE INTELIGENCIA\nARTIFICIAL O CENTRO NACIONAL DE INTELIGENCIA\nARTIFICIAL O CENIA\nDOMICILIO : COMUNA DE LAS CONDES\nLAS CONDES\nREGION METROPOLITANA\nNATURALEZA : CORPORACION\nFECHA CONCESIÓN PJ : 08-02-2022\nDECRETO/RESOLUCIÓN :\nESTADO PJ : VIGENTE\nDIRECTORIO\nÚLTIMA ELECCIÓN DIRECTIVA : 08-02-2022\nDURACIÓN DIRECTIVA : 5 AÑOS\nC

In [ ]:
import json
import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from pathlib import Path

import os
# Esto le dice a macOS que le permita a PyTorch usar toda la memoria unificada disponible
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0" 

def construir_indice_vectorial(archivo_metadata: str, archivo_indice: str, batch_size: int = 32):
    print("🚀 Iniciando el motor de vectorización...")

    dispositivo = "cpu"
    print("⚠️ Advertencia: Corriendo en CPU.")

    # 2. Cargar el modelo BGE-M3
    print(f"📥 Cargando modelo BAAI/bge-m3 en {dispositivo.upper()}... (Esto puede tomar un momento la primera vez)")
    modelo = SentenceTransformer("BAAI/bge-m3", device=dispositivo)
    dimension_vector = modelo.get_sentence_embedding_dimension()
    print(f"✅ Modelo cargado. Dimensión del vector: {dimension_vector}")

    # 3. Leer los fragmentos del archivo metadata.jsonl
    textos_a_vectorizar = []
    print(f"📖 Leyendo fragmentos de {archivo_metadata}...")
    
    with open(archivo_metadata, "r", encoding="utf-8") as f:
        for linea in f:
            if linea.strip():
                datos = json.loads(linea)
                # Extraemos el texto que el modelo leerá
                textos_a_vectorizar.append(datos["texto"])
                
    total_chunks = len(textos_a_vectorizar)
    print(f"📊 Total de fragmentos a procesar: {total_chunks}")

    # 4. Inicializar el índice FAISS (Distancia L2 / Producto Interno)
    # Usamos IndexFlatIP porque los modelos BGE se benefician de la similitud del coseno
    indice_faiss = faiss.IndexFlatIP(dimension_vector)

    # 5. Generar vectores en lotes (Batches) para no saturar la memoria RAM
    print(f"🧠 Generando embeddings en lotes de {batch_size}...")
    
    for i in range(0, total_chunks, batch_size):
        lote_textos = textos_a_vectorizar[i : i + batch_size]
        
        # Generar embeddings
        # normalize_embeddings=True es crucial para usar IndexFlatIP (Similitud del Coseno)
        vectores = modelo.encode(lote_textos, normalize_embeddings=True, show_progress_bar=False)
        
        # Convertir a float32 (FAISS es estricto con los tipos de datos)
        vectores_np = np.array(vectores).astype('float32')
        
        # Añadir al índice FAISS
        indice_faiss.add(vectores_np)
        
        # Imprimir progreso cada 500 chunks
        if (i + len(lote_textos)) % 500 == 0 or (i + len(lote_textos)) == total_chunks:
            print(f"   ⏳ Progreso: {i + len(lote_textos)} / {total_chunks} fragmentos vectorizados.")

    # 6. Guardar el índice en el disco
    print(f"💾 Guardando el índice FAISS en: {archivo_indice}")
    faiss.write_index(indice_faiss, archivo_indice)
    print("🎉 ¡Base de conocimiento construida con éxito!")

# =========================================================
# EJECUCIÓN
# =========================================================
if __name__ == "__main__":
    archivo_origen = "metadata.jsonl"
    archivo_destino = "base_vectorial.index"
    
    construir_indice_vectorial(
        archivo_metadata=archivo_origen, 
        archivo_indice=archivo_destino,
        batch_size=8 # Puedes subirlo a 64 si quieres exigirle un poco más a la memoria
    )


🚀 Iniciando el motor de vectorización...
⚡ Aceleración MPS detectada. Usando la GPU nativa.
📥 Cargando modelo BAAI/bge-m3 en MPS... (Esto puede tomar un momento la primera vez)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
import json
import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from pathlib import Path

def probar_busqueda_faiss(consulta: str, archivo_indice: str, archivo_metadata: str, top_k: int = 3):
    print("🚀 Iniciando motor de búsqueda...")
    
    # 1. Configurar la aceleración por hardware
    if torch.backends.mps.is_available():
        dispositivo = "mps"
    elif torch.cuda.is_available():
        dispositivo = "cuda"
    else:
        dispositivo = "cpu"
        
    print(f"📥 Cargando modelo BAAI/bge-m3 en {dispositivo.upper()} para vectorizar la pregunta...")
    modelo = SentenceTransformer("BAAI/bge-m3", device=dispositivo)
    
    # 2. Cargar Índice FAISS y el archivo de Metadatos
    print("📂 Cargando base vectorial y metadatos...")
    indice = faiss.read_index(archivo_indice)
    
    metadatos = []
    with open(archivo_metadata, 'r', encoding='utf-8') as f:
        for linea in f:
            if linea.strip():
                metadatos.append(json.loads(linea))
                
    # 3. Vectorizar la consulta
    print(f"\n🎯 Buscando: '{consulta}'\n")
    # Es VITAL usar normalize_embeddings=True para que coincida con cómo guardamos los datos
    vector_consulta = modelo.encode([consulta], normalize_embeddings=True, show_progress_bar=False)
    vector_consulta_np = np.array(vector_consulta).astype('float32')
    
    # 4. Buscar en FAISS
    # 'distancias' es qué tan parecido es matemáticamente. 'indices' es el número de fila en tu JSONL
    distancias, indices = indice.search(vector_consulta_np, top_k)
    
    # 5. Imprimir los resultados
    print("==================================================")
    print(f"🏆 TOP {top_k} RESULTADOS ENCONTRADOS")
    print("==================================================\n")
    
    for i in range(top_k):
        idx_resultado = indices[0][i]
        similitud = distancias[0][i]
        
        if idx_resultado != -1:
            resultado_meta = metadatos[idx_resultado]
            print(f"🥇 RELEVANCIA #{i+1} (Score: {similitud:.4f})")
            print(f"📄 Documento:  {resultado_meta.get('doc_id', 'N/A')}")
            print(f"📂 Fuente:     {resultado_meta.get('fuente', 'N/A')}")
            if 'url' in resultado_meta:
                print(f"🔗 URL:        {resultado_meta['url']}")
            print(f"📝 Texto:\n   {resultado_meta.get('texto', '')}")
            print("-" * 70 + "\n")

# =========================================================
# ZONA DE PRUEBAS
# =========================================================
if __name__ == "__main__":
    # Ajusta las rutas a tu estructura final de la entrega
    ruta_indice = "./entrega/base_vectorial/encoder_bge-m3/index.faiss"
    ruta_metadata = "./entrega/base_vectorial/encoder_bge-m3/metadata.jsonl"
    
    # Cambia esto por cualquier pregunta técnica sobre los documentos que procesaste
    pregunta = "¿Cuáles son los riesgos asociados a la basura espacial en órbitas bajas?"
    
    probar_busqueda_faiss(
        consulta=pregunta, 
        archivo_indice=ruta_indice, 
        archivo_metadata=ruta_metadata, 
        top_k=3  # Cambia este número para traer más o menos párrafos
    )
